In [1]:
%matplotlib notebook
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ==================== 参数配置 ====================
# 立方体原始顶点 (中心在原点，边长为2，范围 -1 到 1)
vertices = np.array([
    [-1, -1, -1],  # 0
    [ 1, -1, -1],  # 1
    [ 1, -1,  1],  # 2
    [-1, -1,  1],  # 3
    [-1,  1, -1],  # 4
    [ 1,  1, -1],  # 5
    [ 1,  1,  1],  # 6
    [-1,  1,  1]   # 7
])

# 定义12个三角形面 (每个矩形面拆分为2个三角形，确保渲染正确)
# 格式: [v0, v1, v2]
faces = [
    # 底面 (y = -1)
    [0,1,2], [0,2,3],
    # 顶面 (y = 1)
    [4,6,5], [4,7,6],
    # 前面 (z = 1)
    [3,2,6], [3,6,7],
    # 后面 (z = -1)
    [0,5,1], [0,4,5],
    # 左面 (x = -1)
    [0,3,7], [0,7,4],
    # 右面 (x = 1)
    [1,5,6], [1,6,2]
]

# 颜色: 每个面一种颜色 (RGBA, 半透明)
face_colors = [
    [0.2, 0.6, 0.8, 0.6],  # 底面 蓝
    [0.2, 0.6, 0.8, 0.6],
    [0.8, 0.2, 0.6, 0.6],  # 顶面 粉紫
    [0.8, 0.2, 0.6, 0.6],
    [0.3, 0.8, 0.3, 0.6],  # 前面 绿
    [0.3, 0.8, 0.3, 0.6],
    [0.9, 0.7, 0.1, 0.6],  # 后面 橙黄
    [0.9, 0.7, 0.1, 0.6],
    [0.9, 0.3, 0.3, 0.6],  # 左面 红
    [0.9, 0.3, 0.3, 0.6],
    [0.5, 0.3, 0.9, 0.6],  # 右面 紫罗兰
    [0.5, 0.3, 0.9, 0.6]
]

# ==================== 剪切矩阵生成函数 ====================
def shear_matrix(axis, factor):
    """
    生成剪切变换矩阵 (4x4，用于齐次坐标)
    axis: 'x', 'y', 'z' 指定沿哪个轴方向进行剪切 (保持该轴坐标不变，其他两个轴受影响)
    factor: 剪切因子 (例如沿X轴剪切: x' = x, y' = y + factor * x, z' = z)

    具体:
    - 沿X轴剪切: 保持x不变, 改变y和z: y' = y + sh_xy * x, z' = z + sh_xz * x
    - 沿Y轴剪切: 保持y不变, 改变x和z: x' = x + sh_yx * y, z' = z + sh_yz * y
    - 沿Z轴剪切: 保持z不变, 改变x和y: x' = x + sh_zx * z, y' = y + sh_zy * z

    为了让动画更直观，我们让 factor 同时影响两个相关分量（使用不同比例或相同）
    这里采用: 对于沿X轴剪切: y 变化 factor * x, 同时 z 变化 0.6 * factor * x
              沿Y轴剪切: x 变化 factor * y, z 变化 0.6 * factor * y
              沿Z轴剪切: x 变化 factor * z, y 变化 0.8 * factor * z
    这样动画效果更丰富。
    """
    M = np.eye(4)
    if axis == 'x':
        # 保持 x 不变，y 和 z 随 x 线性变化
        M[1, 0] = factor          # y' = y + factor * x
        M[2, 0] = 0.6 * factor    # z' = z + 0.6*factor * x
    elif axis == 'y':
        # 保持 y 不变，x 和 z 随 y 线性变化
        M[0, 1] = factor          # x' = x + factor * y
        M[2, 1] = 0.6 * factor    # z' = z + 0.6*factor * y
    elif axis == 'z':
        # 保持 z 不变，x 和 y 随 z 线性变化
        M[0, 2] = factor          # x' = x + factor * z
        M[1, 2] = 0.8 * factor    # y' = y + 0.8*factor * z
    return M

def apply_transform(vertices, matrix):
    """对顶点应用4x4变换矩阵 (齐次坐标)"""
    # 添加齐次坐标 (w=1)
    ones = np.ones((vertices.shape[0], 1))
    verts_homo = np.hstack([vertices, ones])  # N x 4
    # 变换
    transformed = verts_homo @ matrix.T       # (N,4) @ (4,4) -> (N,4)
    # 转回三维坐标 (忽略齐次)
    return transformed[:, :3]

# ==================== 动画设置 ====================
# 创建图形和3D轴
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_xlim([-2.5, 2.5])
ax.set_ylim([-2.5, 2.5])
ax.set_zlim([-2.5, 2.5])
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("剪切矩阵动画 (Shear Matrix Transformation)", fontsize=14, pad=20)

# 添加辅助网格和背景样式
ax.grid(True, alpha=0.3)
ax.set_facecolor('#111122')
fig.patch.set_facecolor('#0a0a2a')

# 设置相机视角 (仰角，方位角)
ax.view_init(elev=25, azim=-45)

# 文本信息：显示当前剪切轴和因子
info_text = ax.text2D(0.05, 0.92, "", transform=ax.transAxes,
                      fontsize=12, color='cyan',
                      bbox=dict(facecolor='black', alpha=0.6, boxstyle='round,pad=0.3'))

# 绘制原始立方体 (灰色线框，半透明，用于对比)
def plot_wireframe(ax, verts, color='gray', alpha=0.2):
    """绘制线框"""
    edges = [
        [0,1], [1,2], [2,3], [3,0],  # 底面
        [4,5], [5,6], [6,7], [7,4],  # 顶面
        [0,4], [1,5], [2,6], [3,7]   # 垂直棱
    ]
    for edge in edges:
        ax.plot3D(verts[edge, 0], verts[edge, 1], verts[edge, 2],
                  color=color, alpha=alpha, linewidth=0.8)

# 初始化: 绘制原始立方体线框 (保持一直显示作为参考)
wireframe_verts = vertices.copy()
plot_wireframe(ax, wireframe_verts, color='#88aaff', alpha=0.4)

# 创建动态立方体的三角形面集合 (将被更新)
poly_collection = Poly3DCollection([], alpha=0.75, edgecolor='w', linewidth=0.5)
ax.add_collection3d(poly_collection)

# 动画参数: 循环剪切轴顺序 X -> Y -> Z -> 回到X, 并且因子在 [-1.2, 1.2] 之间平滑振荡
# 使用时间参数 t (0 到 2*pi 循环) 来控制 factor 和当前轴
# 总动画时长 20 秒，200帧
frames = 240
def animate(frame):
    # 周期参数: 0 到 2*pi
    t = 2 * np.pi * frame / frames
    # 在 X, Y, Z 轴之间循环切换 (每个轴主导一个相位区间)
    # 每个轴占据 2/3 pi 的有效展示区间? 为了平滑过渡, 使用正弦绝对值或分段, 但为了清晰演示,
    # 我们让 factor 的正弦因子乘以轴选择权重: 定义三个轴的混合权重, 但实际上我们希望单独展示每个轴的清晰剪切效果,
    # 更好的方式: 分段动画: 0~1/3周期: X轴剪切, 1/3~2/3: Y轴剪切, 2/3~1: Z轴剪切, 同时每个周期内 factor 从 0 上升到最大再回到0.
    # 但为了平滑且避免突变, 使用三个窗口函数: 在每个轴作用区间, factor 从0到峰值再归零, 利用正弦形状.

    # 定义相位区间 [0, 2pi) 映射到 0~1
    phase = t / (2 * np.pi)  # 0-1循环
    # 三个轴的权重因子 (高斯窗口或三角窗口), 使动画过渡自然, 每个轴主导一段
    # 使用上升余弦形状, 轴切换时交叉淡变
    # 轴1: X轴 中心在 phase=0.166, 宽度0.33; 轴2: Y轴 中心=0.5; 轴3: Z轴 中心=0.833
    def gaussian_window(p, center, width=0.28):
        d = (p - center) % 1.0
        if d > 0.5: d = 1.0 - d
        return np.exp(- (d**2) / (2 * (width/2)**2))

    w_x = gaussian_window(phase, 0.166, 0.28)
    w_y = gaussian_window(phase, 0.5, 0.28)
    w_z = gaussian_window(phase, 0.833, 0.28)

    # 归一化权重 (使总活跃度保持连贯, 但不严格必须, 为了视觉平滑)
    # 动态选择主要轴: 选择权重最大的轴, 但同时 factor 按该轴权重缩放峰值.
    # 为了清晰的单独剪切效果, 我们让最大轴主导, 其他轴贡献为0, 但这样会有突变.
    # 更美观: 根据权重混合变换矩阵? 混合矩阵会得到复杂效果可能偏离“剪切”教学.
    # 采用分支: 确定当前主要轴, 因子由该轴的权重和总体正弦调制得到.
    # 为了使动画平滑切换轴, 因子从0上升至峰值再降为0, 不同轴依次执行.
    # 修改策略: 使用分段正弦, 每段内factor从0到1.2再回0, 同时更新轴.
    # 实现方式: 在每1/3周期内, 因子按正弦变化。
    total_segments = 3
    seg_len = frames / total_segments  # 每段帧数
    seg_index = int(frame // seg_len) % total_segments
    local_t = (frame % seg_len) / seg_len  # 0->1 段内进度
    # 因子变化: 正弦形状 0->1->0
    factor = np.sin(np.pi * local_t) * 1.2  # 峰值1.2, 剪切视觉效果明显但不过分

    if seg_index == 0:
        axis = 'x'
        axis_name = "X轴剪切 (Shear along X)"
    elif seg_index == 1:
        axis = 'y'
        axis_name = "Y轴剪切 (Shear along Y)"
    else:
        axis = 'z'
        axis_name = "Z轴剪切 (Shear along Z)"

    # 应用剪切矩阵
    shear_mat = shear_matrix(axis, factor)
    transformed_verts = apply_transform(vertices, shear_mat)

    # 更新多边形集合
    polygons = []
    for i, face in enumerate(faces):
        verts_face = transformed_verts[face, :]
        polygons.append(verts_face)
    poly_collection.set_verts(polygons)
    # 设置颜色 (保持不变)
    poly_collection.set_facecolor(face_colors)

    # 更新文本信息
    info_text.set_text(f"{axis_name}\n剪切因子 (factor) = {factor:.3f}")

    # 动态调整视角轻微旋转，增加动感 (可选，绕Y轴微旋)
    # 不强制修改视角防止眩晕，但可轻微移动相机仰角?
    # 保持视角稳定更好观察立方体形状变化
    # 为了提供更好的视觉效果, 可以稍微根据轴变换改变视角，但保持稳定更利于理解剪切。
    # 保持不变即可。

    # 返回需要更新的对象
    return poly_collection, info_text

# 创建动画
ani = FuncAnimation(fig, animate, frames=frames, interval=50, blit=False, repeat=True)
# 全局配置
plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置中文字体
# 显示动画
plt.tight_layout()
plt.show()

# 如果需要保存为GIF (取消注释下面代码，需安装 pillow 或 ffmpeg)
# 注意: 保存GIF可能需要较长时间，并且文件较大
# ani.save('shear_matrix_animation.gif', writer='pillow', fps=20, dpi=100)

<IPython.core.display.Javascript object>